In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import copy

2025-12-06 10:00:08.191993: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-06 10:00:08.192028: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-06 10:00:08.193282: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-06 10:00:08.201057: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-06 10:00:09.177065: W tensorflow/compiler/tf2

In [2]:
batch_size = 256
learning_rate = 0.001

In [3]:
@tf.keras.saving.register_keras_serializable()
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense2 = tf.keras.layers.Dense(units=1024, activation=tf.nn.leaky_relu)
        self.dense3 = tf.keras.layers.Dense(units=512, activation=tf.nn.leaky_relu)
        self.dense4 = tf.keras.layers.Dense(units=256, activation=tf.nn.leaky_relu)
        self.dense5 = tf.keras.layers.Dense(units=8)

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        x = self.dense3(x)
        x = self.dense4(x)
        output = self.dense5(x)
        return output

In [4]:
class ParaServer:
    def __init__(self):
        self.model = MLP()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    def upload(self, grads):
        self.optimizer.apply_gradients(grads_and_vars=zip(grads, self.model.variables))
        return self.model
    def download(self):
        return self.model
    def initModel(self, x):
        self.model(x)

In [5]:
def valiAll():
    m = ps.download()
    model = copy.deepcopy(m)
    y_v_p = model(X_v)
    va_mse = tf.reduce_mean(tf.square(y_v_p - y_v))
    va_rmse = tf.sqrt(va_mse)
    va_mae = tf.reduce_mean(tf.abs(y_v_p - y_v))
    va_r2 = 1 - tf.reduce_sum(tf.square(y_v_p - y_v)) / tf.reduce_sum(tf.square(y_v - tf.reduce_mean(y_v)))
    print("mse:{} rmse:{} mae:{} r2:{}".format(va_mse, va_rmse, va_mae, va_r2))
    r2sv.append(va_r2.numpy())

In [6]:
class Node:
    def __init__(self, dsName,freq, mu=1e-4):
        self.model = MLP()
        self.freq = freq
        self.mu = mu
        dataset = pd.read_csv(dsName, encoding='utf-8').sample(frac=1).reset_index(drop=True)
        self.X = dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
        self.y = dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)
        self.dataset_train = tf.data.Dataset.from_tensor_slices((self.X, self.y))
        self.dataset_train = self.dataset_train.shuffle(buffer_size=23000)
        self.dataset_train = self.dataset_train.batch(batch_size)
        self.dataset_train = self.dataset_train.prefetch(tf.data.experimental.AUTOTUNE)
    def train(self, num_epochs):
        m = ps.download()
        self.model = copy.deepcopy(m)
        global_weights = [tf.identity(w) for w in self.model.trainable_variables]
        for epoch_index in range(num_epochs):
            for X, y in self.dataset_train:
                with tf.GradientTape() as tape:
                    y_pred = self.model(X)
                    tr_mse = tf.reduce_mean(tf.square(y_pred - y))
                    prox_term = tf.add_n([
                        tf.nn.l2_loss(w - w0)
                        for w, w0 in zip(self.model.trainable_variables, global_weights)
                    ])
                    loss = tr_mse + self.mu * prox_term
                tr_rmse = tf.sqrt(tr_mse)
                tr_mae = tf.reduce_mean(tf.abs(y_pred - y))
                tr_r2 = 1 - tf.reduce_sum(tf.square(y_pred - y)) / tf.reduce_sum(tf.square(y - tf.reduce_mean(y)))
                grads = tape.gradient(loss, self.model.variables)
                m = ps.upload(grads)
                self.model = copy.deepcopy(m)
                # if epoch_index in np.arange(0, num_epochs, 25).tolist() or epoch_index == num_epochs - 1:
            if True:
                print("node:{} epoch:{}".format(self.freq, epoch_index))
                print("train mse:{} rmse:{} mae:{} r2:{}".format(tr_mse, tr_rmse, tr_mae, tr_r2))
                r2s.append(tr_r2.numpy())
                valiAll()

In [7]:
r2s = []
r2sv = []

In [8]:
test_dataset = pd.read_csv("Test.csv", encoding='utf-8').sample(frac=1).reset_index(drop=True)
X_v = test_dataset.loc[:,'freq':'L4'].to_numpy(dtype = np.float32)
y_v = test_dataset.loc[:,'S11r':'S41i'].to_numpy(dtype = np.float32)

In [9]:
ps = ParaServer()
ps.initModel(X_v)

2025-12-06 10:00:10.228323: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21505 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:02:00.0, compute capability: 8.9


In [10]:
nodeList = [Node('./24Train.csv', 2.4), Node('./25Train.csv', 2.5), Node('./26Train.csv', 2.6)]

In [11]:
nodeList[0].train(100)
nodeList[1].train(100)
nodeList[2].train(100)
nodeList[0].train(100)
nodeList[1].train(100)
nodeList[2].train(100)

2025-12-06 10:00:11.802821: I external/local_xla/xla/service/service.cc:168] XLA service 0xb87c2f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-12-06 10:00:11.802868: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2025-12-06 10:00:11.809476: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-12-06 10:00:11.830802: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
I0000 00:00:1765015211.978033  625147 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


node:2.4 epoch:0
train mse:0.08781930059194565 rmse:0.29634320735931396 mae:0.24183966219425201 r2:0.2766057252883911
mse:0.09803277254104614 rmse:0.3131018579006195 mae:0.2534356117248535 r2:0.18849629163742065
node:2.4 epoch:1
train mse:0.08553384989500046 rmse:0.29246169328689575 mae:0.2321341186761856 r2:0.29756927490234375
mse:0.0840974971652031 rmse:0.2899956703186035 mae:0.2351180613040924 r2:0.3038508892059326
node:2.4 epoch:2
train mse:0.07752235978841782 rmse:0.2784283757209778 mae:0.226345032453537 r2:0.35182321071624756
mse:0.0879477709531784 rmse:0.2965599000453949 mae:0.23813146352767944 r2:0.2719787359237671
node:2.4 epoch:3
train mse:0.07340911775827408 rmse:0.2709411680698395 mae:0.22067755460739136 r2:0.3894849419593811
mse:0.08234341442584991 rmse:0.28695541620254517 mae:0.23084332048892975 r2:0.3183709383010864
node:2.4 epoch:4
train mse:0.08604620397090912 rmse:0.29333633184432983 mae:0.2326633781194687 r2:0.2931184768676758
mse:0.08183842897415161 rmse:0.286074161

In [15]:
for i in r2sv:
    print(i)

0.18849629
0.3038509
0.27197874
0.31837094
0.32255113
0.37958068
0.31773323
0.39562017
0.41445816
0.41317886
0.4260429
0.43904305
0.4236794
0.44630682
0.4309941
0.474213
0.44974333
0.46549344
0.5038229
0.4838884
0.5256332
0.47303814
0.5585973
0.4740646
0.46900237
0.4969154
0.41757327
0.45866638
0.5569509
0.38393956
0.48118103
0.34773558
0.40861565
0.52471733
0.5175353
0.4003201
0.49366426
0.37371314
0.30610955
0.39650643
0.44628435
0.50670505
0.53081685
0.42054737
0.4779756
0.48370695
0.5954398
0.5330442
0.47281206
0.4234262
0.42993385
0.31536454
0.5069568
0.35057288
0.46600777
0.41316307
0.36502713
0.46203375
0.4773454
0.34629345
0.41176742
0.48359942
0.5531508
0.44120032
0.4200824
0.2641825
0.536635
0.46406865
0.4805308
0.28575313
0.49441528
0.39928365
0.40684766
0.48248696
0.3585605
0.35303807
0.37463915
0.43113238
0.37523425
0.37715244
0.40089202
0.5350449
0.4287997
0.39353013
0.40652096
0.33723766
0.43820906
0.43368113
0.40396422
0.42171526
0.4261468
0.46067065
0.5235274
0.4552036